# OpenShift S2I Local Directory Deployer (Java / PHP / Ruby)

This notebook helps you take **a local source-code directory** (Java, PHP, or Ruby) and deploy it to **OpenShift** using **Source-to-Image (S2I)** with a **binary build** (`oc start-build --from-dir`).

It will:
- Detect the application language (or you can force it).
- Optionally generate/merge a `.s2i/environment` file with sensible defaults.
- Create an OpenShift **binary S2I BuildConfig** and run a build from your local directory.
- Create a Deployment/Service and optionally a Route.

> **Why binary builds?** `oc new-app <builder>~<git>` expects Git sources. For local sources, OpenShift supports **Binary** input, started with `oc start-build --from-dir`.

---

## Prerequisites
- `oc` CLI installed and on your PATH
- Logged into your cluster: `oc login ...`
- Permissions to create builds and apps in a project/namespace
- Cluster has standard **sample imagestreams** enabled *or* you have access to pull builder images

---

## Safety
This notebook can execute `oc` commands. Keep `DRY_RUN=True` while you review commands.


In [ ]:
# =========================
# User parameters
# =========================
from pathlib import Path

# REQUIRED: local directory with your application sources
SOURCE_DIR = Path("/path/to/your/app").expanduser().resolve()

# Optional: let the notebook detect language. If set, must be one of: "java", "php", "ruby"
LANGUAGE = None  # e.g. "java"

# OpenShift project/namespace where you want to deploy
NAMESPACE = "s2i-demo"

# Application name (used for BuildConfig, ImageStream, Deployment, Service, Route)
APP_NAME = "myapp"

# Whether to actually run oc commands. Start with True to preview.
DRY_RUN = True

# If True, notebook will create/merge .s2i/environment based on detected language
WRITE_S2I_ENV = True

# Builder selection:
# Prefer ImageStreamTag if present in the cluster; otherwise fallback to public pullspec.
# You can override either of these.
DEFAULT_BUILDERS = {
    "java": {
        "imagestreamtag": "openjdk:17",
        "fallback_pullspec": "registry.access.redhat.com/ubi9/openjdk-17",
    },
    "php": {
        "imagestreamtag": "php:8.2",
        "fallback_pullspec": "registry.access.redhat.com/ubi9/php-82",
    },
    "ruby": {
        "imagestreamtag": "ruby:3.1",
        "fallback_pullspec": "registry.access.redhat.com/ubi9/ruby-31",
    },
}

# Route exposure (typical for web apps). You can disable.
EXPOSE_ROUTE = True

# If you need to set runtime env vars on the Deployment, list them here:
RUNTIME_ENV = {
    # "JAVA_OPTS_APPEND": "-Dquarkus.http.host=0.0.0.0",
}


In [ ]:
import shlex, subprocess
from pathlib import Path
from typing import Dict, Optional, Tuple

def run(cmd: str, check: bool=True) -> Tuple[int, str]:
    # Run a shell command. Returns (rc, combined_output). Honors DRY_RUN.
    print(f"\n$ {cmd}")
    if DRY_RUN:
        return 0, "(dry-run) command not executed"
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed (rc={p.returncode}): {cmd}\n\n{p.stdout}")
    return p.returncode, p.stdout

def detect_language(src: Path) -> str:
    # Detect language using common project markers.
    markers = {
        "java": ["pom.xml", "build.gradle", "build.gradle.kts", "mvnw", "gradlew"],
        "php":  ["composer.json", "index.php", "artisan", "symfony.lock"],
        "ruby": ["Gemfile", "config.ru", "Rakefile", "bin/rails"],
    }
    hits = {lang: 0 for lang in markers}
    for lang, files in markers.items():
        for f in files:
            if (src / f).exists():
                hits[lang] += 1
    best = sorted(hits.items(), key=lambda kv: kv[1], reverse=True)[0]
    if best[1] == 0:
        raise ValueError(
            f"Could not detect language for {src}. "
            "Set LANGUAGE explicitly to one of: java|php|ruby."
        )
    return best[0]

def ensure_s2i_env(src: Path, env: Dict[str, str]) -> Path:
    # Create or merge .s2i/environment (key=value lines).
    s2i_dir = src / ".s2i"
    s2i_dir.mkdir(parents=True, exist_ok=True)
    env_file = s2i_dir / "environment"

    existing = {}
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            existing[k.strip()] = v.strip()

    merged = dict(existing)
    merged.update(env)

    content = "\n".join([f"{k}={v}" for k, v in merged.items()]) + "\n"
    env_file.write_text(content)
    return env_file

def oc(*args: str, check: bool=True):
    cmd = "oc " + " ".join(shlex.quote(a) for a in args)
    return run(cmd, check=check)

def check_oc_login() -> None:
    rc, out = oc("whoami", check=False)
    if rc != 0:
        raise RuntimeError("You are not logged in. Run `oc login ...` and re-run this cell.")
    print(f"Logged in as: {out.strip()}")

def has_imagestreamtag(ist: str) -> bool:
    rc, _ = oc("-n", "openshift", "get", "istag", ist, "-o", "name", check=False)
    return rc == 0


In [ ]:
assert SOURCE_DIR.exists() and SOURCE_DIR.is_dir(), f"SOURCE_DIR does not exist or is not a directory: {SOURCE_DIR}"

lang = LANGUAGE or detect_language(SOURCE_DIR)
print(f"Detected language: {lang}")

builder_ist = DEFAULT_BUILDERS[lang]["imagestreamtag"]
builder_pullspec = DEFAULT_BUILDERS[lang]["fallback_pullspec"]

print(f"Preferred builder ImageStreamTag: {builder_ist}")
print(f"Fallback builder pullspec:      {builder_pullspec}")

# Basic structure checks & hints
hints = []
if lang == "java":
    if (SOURCE_DIR / "pom.xml").exists():
        hints.append("Java (Maven) detected: pom.xml found. S2I will run Maven build by default.")
    elif (SOURCE_DIR / "build.gradle").exists() or (SOURCE_DIR / "build.gradle.kts").exists():
        hints.append("Java (Gradle) detected: build.gradle* found. Ensure your build outputs a runnable JAR.")
    else:
        hints.append("Java detected but no pom.xml/build.gradle found. Consider adding Maven/Gradle, or use a binary artifact workflow.")
elif lang == "php":
    if (SOURCE_DIR / "composer.json").exists():
        hints.append("PHP detected: composer.json found. Builder typically installs dependencies during assemble.")
    if (SOURCE_DIR / "public").exists():
        hints.append("Found /public directory. Consider setting DOCUMENTROOT=/public in .s2i/environment.")
elif lang == "ruby":
    if (SOURCE_DIR / "Gemfile").exists():
        hints.append("Ruby detected: Gemfile found. Builder typically runs bundle install during assemble.")
    if (SOURCE_DIR / "config.ru").exists():
        hints.append("Rack config.ru found.")

print("\n".join(["- " + h for h in hints]) if hints else "No additional hints.")


In [ ]:
# Optional: write/merge .s2i/environment with common defaults for each language.
# Adjust these defaults to match your app.
if WRITE_S2I_ENV:
    defaults = {}
    if lang == "java":
        defaults = {
            "MAVEN_ARGS_APPEND": "-DskipTests",
            "JAVA_MAX_MEM_RATIO": "80",
        }
    elif lang == "php":
        defaults = {
            # Uncomment if your PHP app serves from /public:
            # "DOCUMENTROOT": "/public",
            "PHP_MEMORY_LIMIT": "256M",
        }
    elif lang == "ruby":
        defaults = {
            "RACK_ENV": "production",
            # "DISABLE_ASSET_COMPILATION": "true",
        }

    env_file = ensure_s2i_env(SOURCE_DIR, defaults)
    print(f"Wrote/merged: {env_file}")
    print(env_file.read_text())
else:
    print("WRITE_S2I_ENV=False, skipping.")


In [ ]:
# =========================
# Deploy with Binary S2I build
# =========================

check_oc_login()

# Create/target project
oc("new-project", NAMESPACE, check=False)
oc("project", NAMESPACE)

# Choose builder: ImageStreamTag (cluster-provided) if available, else pullspec.
if has_imagestreamtag(builder_ist):
    builder = builder_ist
    print(f"Using ImageStreamTag builder: {builder}")
else:
    builder = builder_pullspec
    print(f"Using pullspec builder: {builder}")

# 1) Create BuildConfig for binary S2I
oc("new-build", "--name", APP_NAME, "--binary=true", "--strategy=source", builder)

# 2) Start build from local directory
oc("start-build", APP_NAME, f"--from-dir={str(SOURCE_DIR)}", "--follow")

# 3) Create app from the resulting image stream
oc("new-app", "--image-stream", f"{APP_NAME}:latest", "--name", APP_NAME, check=False)

# 4) Set runtime environment variables (optional)
for k, v in RUNTIME_ENV.items():
    oc("set", "env", f"deploy/{APP_NAME}", f"{k}={v}", check=False)

# 5) Expose Route (optional)
if EXPOSE_ROUTE:
    oc("expose", "svc", APP_NAME, check=False)

# 6) Show status
oc("get", "pods")
oc("get", "svc")
if EXPOSE_ROUTE:
    oc("get", "route", APP_NAME, check=False)


In [ ]:
# Quick smoke test (requires route to exist and your cluster/router to be reachable)
if EXPOSE_ROUTE:
    _, route_out = oc("get", "route", APP_NAME, "-o", "jsonpath={.spec.host}", check=False)
    host = route_out.strip()
    if host:
        print(f"Route host: {host}")
        print("Try: curl -k https://" + host)
    else:
        print("No route host found.")
else:
    print("Route is disabled. Use port-forward or expose a route manually.")


In [ ]:
# Cleanup (optional)
# oc("delete", "project", NAMESPACE, check=False)
